# Government hospital beds per 1,000 population

This notebook replaces raw state-level bed counts with a population-adjusted availability measure using the Central Bureau of Health Intelligence's National Health Profile 2023. Bed counts are as of 31 December 2022. The denominator is the Ministry of Health & Family Welfare's projected population as of 1 March 2021, reported in thousands. The approximately 21-month timing gap is disclosed; rates are indicative, not exact same-date ratios.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../Dataset')
beds = pd.read_csv(DATA / 'state_government_hospital_beds_nhp2023_per_1000.csv')
beds['beds_per_1000_population_recalculated'] = beds['beds_total_nhp2023'] / beds['population_2021_projected_thousands']
beds['rate_difference'] = beds['beds_per_1000_population'] - beds['beds_per_1000_population_recalculated']
assert beds['rate_difference'].abs().max() < 0.00011
print('State/UT observations:', len(beds))
print('Bed count reference date:', beds['beds_reference_date'].unique())
print('Population reference date:', beds['population_reference_date'].unique())
beds.head()

## 1. Compare availability across states and UTs

The numerator covers government hospitals operated by central, state and local government bodies, and excludes private-sector beds. Delhi and Chandigarh are included as urban-only observations in the NHP table. The combined Dadra & Nagar Haveli and Daman & Diu population is summed from the two separate 2021 projections to match the merged bed-count geography.

In [ ]:
ranked = beds.sort_values('beds_per_1000_population', ascending=False).copy()
display(ranked[['state_ut','beds_total_nhp2023','population_2021_projected_thousands','beds_per_1000_population']].round(3))
print('Median beds per 1,000 population:', round(beds['beds_per_1000_population'].median(), 3))
print('Mean beds per 1,000 population:', round(beds['beds_per_1000_population'].mean(), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 11))
plot_data = ranked.sort_values('beds_per_1000_population', ascending=True)
ax.barh(plot_data['state_ut'], plot_data['beds_per_1000_population'])
ax.set_xlabel('Government hospital beds per 1,000 projected population')
ax.set_ylabel('State / Union Territory')
ax.set_title('Government hospital bed availability by state/UT')
fig.tight_layout()
plt.show()

## 2. Data limitations

- The bed-count table is dated 31 December 2022; population is projected as of 1 March 2021. The 21-month gap means the denominator is not contemporaneous.
- The rate covers government hospital beds only; it is not total public-plus-private bed availability.
- State reporting and facility coverage may differ. NHP notes that government hospitals include central, state and local government bodies.
- The ratio is a descriptive capacity indicator, not a measure of geographic accessibility, occupancy, staffing, quality or effective service availability.
- Small UTs may have volatile rates because their populations are small.

In [ ]:
# Reproducibility checks
assert len(beds) == 36
assert beds['state_ut'].is_unique
assert beds['beds_total_nhp2023'].gt(0).all()
assert beds['population_2021_projected_thousands'].gt(0).all()
assert beds['beds_per_1000_population'].between(0, 10).all()
assert beds['rate_difference'].abs().max() < 0.00011
print('Notebook 08 checks passed.')